# Milestone 3: train the inverse model on a GPU

Trains the ~10M-parameter encoder-decoder transformer that maps rotational constants (plus optional formula and dipole) to candidate molecular structures (SELFIES), then evaluates recall@K on held-out molecules whose formulas were never seen in training.

**Runtime:** Runtime -> Change runtime type -> GPU (a T4 is enough). Expected time: ~5 min setup, ~1-2 h training (150 epochs), ~10-20 min evaluation.

If the repository is private, cloning needs a GitHub token (see the clone cell).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

For a private repo, put a fine-grained GitHub token (read access to this repo) in Colab's **Secrets** panel as `GITHUB_TOKEN` and keep `PRIVATE = True`.

In [ ]:
BRANCH = "claude/serene-meitner-x48vgm"
PRIVATE = True

import os, subprocess
url = "https://github.com/Nathan-W123/Quantum-ML.git"
if PRIVATE:
    from google.colab import userdata
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/Nathan-W123/Quantum-ML.git"
if not os.path.exists("Quantum-ML"):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "Quantum-ML"], check=True)
%cd Quantum-ML
!git log --oneline -3

In [ ]:
!pip install -q -e ".[ml]"

## 2. Data (QM9 from the MoleculeNet mirror)

`01_prepare_qm9.py` re-perceives bonds from the 3D geometries (the SDF's bond orders are wrong for ~19% of molecules). `04_build_m3_dataset.py` makes the formula-split dataset.

In [ ]:
!mkdir -p data/raw/qm9_molnet
!curl -sL -o data/raw/qm9_molnet/qm9.zip https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/molnet_publish/qm9.zip
!cd data/raw/qm9_molnet && unzip -oq qm9.zip
!python scripts/01_prepare_qm9.py
!python scripts/04_build_m3_dataset.py

## 3. Train

Checkpoints go to Google Drive so a disconnect does not lose the run. If the session dies, re-run the cells above and then this one: `--resume` continues from `last.pt`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN = "/content/drive/MyDrive/quantum-ml/runs/m3_baseline"

In [ ]:
!python scripts/05_train_m3.py --config configs/m3_baseline.json --out "$RUN" --resume

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
h = pd.read_csv(f"{RUN}/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
h.plot(x="epoch", y=["train_ce", "val_ce"], ax=ax[0], title="Cross-entropy")
cols = [c for c in h.columns if "recall" in c]
h.dropna(subset=cols).plot(x="epoch", y=cols, ax=ax[1], title="Validation recall (beam 10, 0.5% noise)")
plt.show()

## 4. Evaluate on held-out formulas

Recall@K for 2,000 test molecules at 0.1%, 0.5% and 1% noise, for each information condition, with 1,000 samples + beam 10 per molecule. The `library` columns are the reference from ranking all QM9 molecules (including the answer) by chi-square fit.

In [ ]:
!python scripts/06_eval_m3.py --run "$RUN" --n-eval 2000 --samples 1000 --batch 16

In [ ]:
s = pd.read_csv(f"{RUN}/eval/summary.csv")
cols = ["noise_pct", "condition"] + [c for c in s.columns if c.startswith(("model recall", "library recall"))]
s[cols].round(3)